1. Install

In [4]:
pip install transformers sentence-transformers faiss-cpu gradio

2. Import libraries

In [5]:
import gradio as gr
import faiss
import numpy as np
from transformers import pipeline
from sentence_transformers import SentenceTransformer


3. Load Models

In [6]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
qa_pipeline = pipeline("question-answering",
                        model="deepset/roberta-base-squad2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


4. Build Vector Index from Document

In [7]:
def build_index(text):
    # Split into chunks
    chunks = [text[i:i+300] for i in range(0, len(text), 250)]

    # Embed chunks
    embeddings = embedder.encode(chunks)
    embeddings = np.array(embeddings).astype("float32")

    # Build FAISS index
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)

    return index, chunks


5. Retrieve + Answer

In [8]:
def answer_question(document, question):
    if not document or not question:
        return "Please provide both a document and a question."

    index, chunks = build_index(document)

    # Embed the question
    q_embedding = embedder.encode([question]).astype("float32")

    # Retrieve top 3 chunks
    _, indices = index.search(q_embedding, k=3)
    context = " ".join([chunks[i] for i in indices[0]])

    # Generate answer
    result = qa_pipeline(question=question, context=context)

    return f"**Answer:** {result['answer']}\n\n**Confidence:** {round(result['score']*100, 2)}%\n\n**Source chunk:** ...{context[:300]}..."


6. Gradio UI

In [9]:
demo = gr.Interface(
    fn=answer_question,
    inputs=[
        gr.Textbox(lines=15, placeholder="Paste your document here...", label="📄 Document"),
        gr.Textbox(lines=2, placeholder="Ask a question about the document...", label="❓ Your Question")
    ],
    outputs=gr.Markdown(label="💡 Answer"),
    title="🤗 RAG-based Document Q&A",
    description="Paste any document and ask questions. Powered by HuggingFace + FAISS.",
    examples=[
        ["Hyderabad is the capital of Telangana state in India. It is known as the City of Pearls and is a major hub for IT companies including Microsoft, Google, and Amazon. The city has a rich history with monuments like Charminar and Golconda Fort.",
         "What is Hyderabad known for?"]
    ]
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5e14b4ff43f7887800.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
